In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift
from utility.my_colors import pink

from sparcl.client import SparclClient
from dl import queryClient as qc
from astropy.table import Table

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [113]:
print(qc.schema('desi_dr1.zpix'))


Schema: desi_dr1
 Table: zpix

     Column Name   Description
     -----------   -----------
     tsnr2_lrg_b   LRG B template (S/N)^2
 tsnr2_gpbdark_r   GPBDARK R template (S/N)^2
     tsnr2_elg_r   ELG R template (S/N)^2
tsnr2_gpbbright_r   GPBBRIGHT R template (S/N)^2
     tsnr2_lya_r   LYA R template (S/N)^2
               z   Redshift measured by Redrock
            zerr   Redshift error from Redrock
            chi2   Best fit chi squared
         coeff_0   Redrock template coefficients
         coeff_1   Redrock template coefficients
         coeff_2   Redrock template coefficients
         coeff_3   Redrock template coefficients
         coeff_4   Redrock template coefficients
         coeff_5   Redrock template coefficients
         coeff_6   Redrock template coefficients
         coeff_7   Redrock template coefficients
         coeff_8   Redrock template coefficients
         coeff_9   Redrock template coefficients
       deltachi2   Delta-chi-squared for template fit from R

In [6]:
query = """
        SELECT zp.targetid, zp.z, zp.zerr, agn.civ_1549_flux, agn.civ_1549_flux_ivar, agn.civ_1549_sigma, agn.flux_w3, agn.flux_ivar_w3, zp.desiname
        FROM desi_dr1.zpix AS zp
        JOIN desi_dr1.agngal as agn
        ON zp.targetid = agn.targetid
        WHERE (zp.z BETWEEN 2.2 AND 4.38) AND (zp.zwarn=0) AND (agn.civ_1549_flux>0) AND (zp.spectype='QSO') AND zp.zcat_primary
        """



# query_agngal = """
#         SELECT targetid, z, zerr, zwarn, civ_1549_flux, civ_1549_flux_ivar, civ_1549_sigma, flux_w3, flux_ivar_w3, spectype, survey
#         FROM desi_dr1.agngal
#         WHERE (zwarn=0) AND main_primary AND (spectype='QSO') AND (z BETWEEN 2.2 AND 4.38) AND (civ_1549_flux > 0)
#         """

# query_zpix = """
#         SELECT zp.targetid, zp.survey, zp.program, zp.desi_target, 
#                zp.mean_fiber_ra, zp.mean_fiber_dec, 
#                zp.z, zp.zwarn, zp.spectype
#         FROM desi_dr1.zpix AS zp
#         WHERE zp.main_primary AND (zp.z BETWEEN 2.2 AND 4.38) AND (zp.spectype='QSO') AND (zp.zwarn=0)
#         """ 

In [ ]:
# results = qc.query(sql=query, fmt="pandas")
# results_agngal = qc.query(sql=query_agngal, fmt="pandas")
# results_zpix = qc.query(sql=query_zpix, fmt="pandas")

In [2]:
path = "data/target_properties.csv"
# results.to_csv(path)

results = pd.read_csv(path,index_col=0)
results.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404142 entries, 0 to 404141
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   targetid            404142 non-null  int64  
 1   z                   404142 non-null  float64
 2   zerr                404142 non-null  float64
 3   civ_1549_flux       404142 non-null  float64
 4   civ_1549_flux_ivar  404142 non-null  float64
 5   civ_1549_sigma      404142 non-null  float64
 6   flux_w3             404142 non-null  float64
 7   flux_ivar_w3        403793 non-null  float64
 8   desiname            404142 non-null  object 
dtypes: float64(7), int64(1), object(1)
memory usage: 30.8+ MB


In [11]:
target_idxs = [int(x) for x in results["targetid"].unique()]

In [ ]:
# results["snr_w3"] = results["flux_w3"] * np.sqrt(results["flux_ivar_w3"])
# results["civ_1549_fwhm"] = results["civ_1549_sigma"] * 2.355

In [15]:
# results.query("snr_w3 > 3 & civ_1549_fwhm > 500 & civ_1549_fwhm < 20000")

In [3]:
client = SparclClient()

announcement=Data set deprecation notice: on November 19, 2025 the SDSS/BOSS DR16 data sets were deprecated. Please use the new SDSS/BOSS DR17 data sets instead.


In [ ]:
# output_fields = ['targetid', 'redshift', 'flux', 'ivar', 'wavelength', 'spectype', 'specprimary','survey', 'redshift_warning','desi_name']

In [77]:
# spectra = client.retrieve_by_specid(specid_list=target_idxs, include=output_fields, dataset_list=["DESI-DR1"])

In [132]:
# spectra_df = pd.DataFrame(spectra.data[1:])
# spectra_df

In [6]:
out = ['sparcl_id', 'targetid', 'flux', 'wavelength', 'ivar', 'ra', 'dec', 'redshift', 'desiname']

# constraints for query
# constraints = {'spectype':['GALAXY'], 'ra':[208.4, 210.2], 'dec':[4.8, 6.4], 'specprimary': [True], 'data_release': ['DESI-DR1'], 'redshift': [0.1, 0.15]}
constraints = {'spectype':['QSO'], 'specprimary': [True], 'data_release': ['DESI-DR1'], 'redshift': [2.2,4.38], 'redshift_warning':[0]}

desi_results = client.find(outfields=out, constraints=constraints, limit=None)


In [24]:
idxs = []
sparcl_idxs = []
for i in range(1,len(desi_results)):
    idxs.append(desi_results[i]["targetid"])
    sparcl_idxs.append([desi_results[i]["targetid"],desi_results[i]["sparcl_id"]])

In [27]:
sparcl_idxs_df = pd.DataFrame(sparcl_idxs,columns=["targetid","sparcl_id"])
sparcl_idxs_df

,targetid,sparcl_id
0,39628463971241314,000004f1-8b6d-11ef-9a2e-525400f334e1
1,39628220877768838,000011ac-8a7c-11ef-bff7-525400f334e1
2,39627977599748859,00006fd7-88d2-11ef-8266-525400f334e1
3,39632941046826799,00008713-865b-11ef-9960-525400f334e1
4,39627896146362734,0000a401-87c2-11ef-a7de-525400f334e1
...,...,...
402007,39628471252549681,ffff53ba-86b7-11ef-bf00-525400f334e1
402008,39627941692312539,ffff89b3-87d9-11ef-9fd6-525400f334e1
402009,39627660766220288,ffff9b26-8745-11ef-9f6f-525400f334e1
402010,39628507344535670,ffffca6e-867b-11ef-a551-525400f334e1


In [41]:
test_idxs = sparcl_idxs_df["sparcl_id"].to_list()[:10]
test = client.retrieve_by_specid(idxs[:500],dataset_list=["DESI-DR1"],limit=None)

In [43]:
test

Retrieved Results: 523 records

In [ ]:
# len(target_idxs)
# len(idxs)
len(set(target_idxs) & set(idxs))

392649